In [1]:
import anndata
import pandas as pd
import plotnine as p
import scvi
import torch

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!


In [2]:
test_sets = {}
test_sets['brain_test'] = anndata.io.read_h5ad('250508.brain_test.GSE212576.h5ad')
test_sets['liver_test'] = anndata.io.read_h5ad('250508.liver_test.GSE166504.h5ad')

In [3]:
models = {}
models['brain_model'] = scvi.model.SCVI.load('250508.brain.model', test_sets['brain_test'])
models['liver_model'] = scvi.model.SCVI.load('250508.liver.model', test_sets['brain_test'])
models['linear_merged_model'] = scvi.model.SCVI.load('250517.linear.merge.model', test_sets['brain_test'])
models['nuslerp_merged_model'] = scvi.model.SCVI.load('250517.nuslerp.merge.model', test_sets['brain_test'])
models['combined_10e_model'] = scvi.model.SCVI.load('250517.combined_model.10e.model', test_sets['brain_test'])
models['combined_20e_model'] = scvi.model.SCVI.load('250517.combined_model.20e.model', test_sets['brain_test'])

INFO     File 250508.brain.model/model.pt already downloaded                                                       


INFO     File 250508.liver.model/model.pt already downloaded                                                       
INFO     File 250517.linear.merge.model/model.pt already downloaded                                                
INFO     File 250517.nuslerp.merge.model/model.pt already downloaded                                               
INFO     File 250517.combined_model.10e.model/model.pt already downloaded                                          
INFO     File 250517.combined_model.20e.model/model.pt already downloaded                                          


In [5]:
results = []
for model in models:
    for test_set in test_sets:
        results += [{
            'model': model,
            'test_set': test_set,
            'reconstruction_error': (
                models[model]
                .get_reconstruction_error(test_sets[test_set])['reconstruction_loss']
                .cpu()
                .numpy()
            )
        }]

INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


In [6]:
df = pd.DataFrame(results)
df

,model,test_set,reconstruction_error
0,brain_model,brain_test,7803.1816
1,brain_model,liver_test,10803.492
2,liver_model,brain_test,36707.863
3,liver_model,liver_test,3688.7688
4,linear_merged_model,brain_test,15792.342
5,linear_merged_model,liver_test,7397.1396
6,nuslerp_merged_model,brain_test,15792.296
7,nuslerp_merged_model,liver_test,7397.611
8,combined_10e_model,brain_test,7833.823
9,combined_10e_model,liver_test,3714.92


In [7]:
pivoted = (
    df
    .pivot_table(
        index = 'model',
        columns = 'test_set',
        values = 'reconstruction_error'
    )
)

In [9]:
desired = [
    "brain_model",
    "liver_model",
    "linear_merged_model",
    "nuslerp_merged_model",
    "combined_10e_model",
    "combined_20e_model",
]

pivoted = pivoted.reindex(desired)

In [10]:
pivoted

test_set,brain_test,liver_test
model,,
brain_model,7803.181641,10803.492188
liver_model,36707.863281,3688.768799
linear_merged_model,15792.341797,7397.139648
nuslerp_merged_model,15792.295898,7397.61084
combined_10e_model,7833.823242,3714.919922
combined_20e_model,7815.274902,3700.787842


In [11]:
df.to_csv('250517.results.csv', index = False)